In [1]:
import warnings

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

FEATURES_GROUP = [
    "base",
    "all_history",
    "rolling_windows",
    "velocity",
    "trend",
    "gmv",
    "conversion_rates",
    "ewm",
]


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


def load_split(split_name: str, feature_name: str) -> pd.DataFrame:
    path = f"datasets/{split_name}_{feature_name}.parquet"
    return pd.read_parquet(path)


def get_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = {"user_id", "target"}
    return [
        c for c in df.columns
        if c not in exclude and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


def merge_splits(split_name: str, feature_names: list[str]) -> pd.DataFrame:
    base = load_split(split_name, feature_names[0])
    for name in feature_names[1:]:
        extra = load_split(split_name, name)
        new_cols = [c for c in extra.columns if c not in base.columns]
        base = base.merge(extra[["user_id"] + new_cols], on="user_id", how="left")
    return base


def train_and_eval(train_df: pd.DataFrame, val_df: pd.DataFrame) -> tuple[float, int]:
    feat_cols = get_feature_cols(train_df)

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=False,
        early_stopping_rounds=50,
    )
    model.fit(
        train_df[feat_cols].values,
        train_df["target"].values,
        eval_set=(val_df[feat_cols].values, val_df["target"].values),
    )
    preds = model.predict(val_df[feat_cols].values)
    score = rmsle(val_df["target"].values, preds)
    return score, model.best_iteration_


# =============================================================================
# Main
# =============================================================================

results = []

print("\n--- Individual feature groups ---")
for feature_name in FEATURES_GROUP:
    print(f"\nFeature group: {feature_name}")
    train_df = load_split("split1", feature_name)
    val1_df = load_split("split2", feature_name)
    val2_df = load_split("split3", feature_name)
    score1, best_iter1 = train_and_eval(train_df, val1_df)
    score2, best_iter2 = train_and_eval(train_df, val2_df)
    print(f" split1 RMSLE = {score1:.5f}  (best iteration: {best_iter1}) | split2 RMSLE = {score2:.5f}  (best iteration: {best_iter2})")
    results.append({"experiment": "split1_" + feature_name , "groups": feature_name, "rmsle": score1})
    results.append({"experiment": "split2_" + feature_name , "groups": feature_name, "rmsle": score2})

print("\n--- Cumulative feature groups ---")
for i in range(2, len(FEATURES_GROUP) + 1):
    subset = FEATURES_GROUP[:i]
    label = " + ".join(subset)
    print(f"\nCumulative: {label}")
    train_df = merge_splits("split1", subset)
    val1_df = merge_splits("split2", subset)
    val2_df = merge_splits("split3", subset)
    score1, best_iter1 = train_and_eval(train_df, val1_df)
    score2, best_iter2 = train_and_eval(train_df, val2_df)
    print(f" split1 RMSLE = {score1:.5f}  (best iteration: {best_iter1}) | split2 RMSLE = {score2:.5f}  (best iteration: {best_iter2})")
    results.append({"experiment": f"split1_cumulative_{i}", "groups": label, "rmsle": score1})
    results.append({"experiment": f"split2_cumulative_{i}", "groups": label, "rmsle": score2})

# =============================================================================
# Summary
# =============================================================================

results_df = pd.DataFrame(results).sort_values("rmsle")

print("\n" + "=" * 80)
print("Ranking by RMSLE (lower = better)")
print("=" * 80)
print(results_df.to_string(index=False))


 split1 RMSLE = 2.45084  (best iteration: 174) | split2 RMSLE = 2.82212  (best iteration: 36)

Ranking by RMSLE (lower = better)
             experiment                                                                                 groups    rmsle
     spli1_cumulative_7       base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates 2.423333
     spli1_cumulative_3                                                   base + all_history + rolling_windows 2.433218
     spli1_cumulative_5                                base + all_history + rolling_windows + velocity + trend 2.434959
     split1_all_history                                                                            all_history 2.435938
     spli1_cumulative_2                                                                     base + all_history 2.435938
     spli1_cumulative_6                          base + all_history + rolling_windows + velocity + trend + gmv 2.437172
     spli1_cumulative_4        

In [19]:
import joblib
import catboost as cb
import optuna

RANDOM_STATE = 42


def get_feat_cols(df):
    return [
        c for c in df.columns
        if c not in ("user_id", "target")
           and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


cb_train = pd.read_parquet("datasets/split1_all_history,rolling_windows,velocity,trend,gmv.parquet")
cb_val = pd.read_parquet("datasets/split2_all_history,rolling_windows,velocity,trend,gmv.parquet")

cb_feat_cols = [c for c in cb_train.columns if
                c not in ("user_id", "target") and not pd.api.types.is_datetime64_any_dtype(cb_train[c])]

print(f"  CatBoost  — train: {cb_train.shape[0]} rows, {len(cb_feat_cols)} features")

X_cb_tr = cb_train[cb_feat_cols].values
y_cb_tr = np.log1p(cb_train["target"].values)
X_cb_val = cb_val[cb_feat_cols].values
y_cb_val = np.log1p(cb_val["target"].values)
y_cb_val_raw = cb_val["target"].values

print("\nOptuna tuning CatBoost (20 trials)...")


def cb_objective(trial):
    params = dict(
        loss_function="RMSE",
        iterations=trial.suggest_int("iterations", 500, 3000),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        depth=trial.suggest_int("depth", 3, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 10),
        min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 1, 50),
        random_seed=RANDOM_STATE,
        verbose=False,
        early_stopping_rounds=50,
    )
    model = cb.CatBoostRegressor(**params)
    model.fit(X_cb_tr, y_cb_tr, eval_set=(X_cb_val, y_cb_val))
    return rmsle(y_cb_val_raw, model.predict(X_cb_val))


cb_study = optuna.create_study(direction="minimize")
cb_study.optimize(cb_objective, n_trials=20)
best_cb_params = cb_study.best_params

print(f"  Best CatBoost RMSLE: {cb_study.best_value:.5f}")
print(f"  Params: {best_cb_params}")

final_cb = cb.CatBoostRegressor(
    **best_cb_params,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    early_stopping_rounds=50,
)
final_cb.fit(X_cb_tr, y_cb_tr, eval_set=(X_cb_val, y_cb_val))

cb_best_n = final_cb.best_iteration_ + 1

X_cb_full = np.concatenate([X_cb_tr, X_cb_val])
y_cb_full = np.concatenate([y_cb_tr, y_cb_val])

cb_sub_params = {k: v for k, v in best_cb_params.items() if k != "iterations"}

final_cb_full = cb.CatBoostRegressor(
    **cb_sub_params,
    iterations=cb_best_n,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
)
final_cb_full.fit(X_cb_full, y_cb_full)

joblib.dump(final_cb, "model/cat_split1.joblib")
joblib.dump(final_cb_full, "model/cat_full.joblib")

KeyboardInterrupt: 